# Testing Azure OpenAI Embeddings with CosmosDB

**This example demonstrates how to use Azure OpenAI for embeddings with CosmosDB as the vector database.**

To use CosmosDB as a toolbox with Azure OpenAI, you will need to complete the following steps:

1. Create an Azure CosmosDB Account:
   - Go to the Azure Portal (https://portal.azure.com).
   - Create a new CosmosDB account with MongoDB API.
   - Configure your database and collection settings.

2. Set Up Azure OpenAI:
   - Create an Azure OpenAI resource in the Azure Portal.
   - Deploy the models you need (e.g., text-embedding-3-small).
   - Get your API key and endpoint URL.

3. Configure Network Access:
   - Configure firewall rules for your CosmosDB account.
   - Allow access from your IP address or configure appropriate network security.

4. Obtain the Connection String:
   - Get the MongoDB connection string from your CosmosDB account.
   - This will be used to connect to your CosmosDB instance.

Now that you have your CosmosDB connection string and Azure OpenAI credentials, you can use them to connect in the `memorizz` library.

In [ ]:
pip install -e .

In [2]:
from memorizz import CosmosDBToolsConfig, CosmosDBTools

In [3]:
import os
import getpass

# Set up environment variables for Azure OpenAI and CosmosDB
# This key is required for using Azure OpenAI's services, such as generating embeddings
AZURE_OPENAI_API_KEY = getpass.getpass("Azure OpenAI API Key: ")
os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY

# This endpoint is needed to connect to Azure OpenAI
AZURE_OPENAI_ENDPOINT = getpass.getpass("Enter Azure OpenAI Endpoint: ")
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT

# This connection string is needed to connect to the CosmosDB database
COSMOSDB_CONNECTION_STRING = getpass.getpass("Enter CosmosDB Connection String: ")
os.environ["COSMOSDB_CONNECTION_STRING"] = COSMOSDB_CONNECTION_STRING

In [10]:
# 1. Initialize the CosmosDB configuration and create a CosmosDB tools instance
from memorizz.embeddings.openai import get_embedding
from memorizz import AzureOpenAIEmbeddingConfig

# Create Azure OpenAI embedding configuration
azure_embed_config = AzureOpenAIEmbeddingConfig(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

# Create embedding function with Azure OpenAI
def azure_embedding_func(text: str):
    return get_embedding(
        text=text,
        model="text-embedding-3-small",
        dimensions=256,
        azure_config=azure_embed_config
    )

config = CosmosDBToolsConfig(
    connection_string=COSMOSDB_CONNECTION_STRING,  # CosmosDB connection string
    db_name="function_calling_new",  # Name of the database to use
    collection_name="tools",  # Name of the collection to store tools
    vector_search_candidates=150,  # Number of candidates to consider in vector search
    vector_index_name="vector_index",  # Name of the vector index in CosmosDB
    get_embedding=azure_embedding_func
)

# 2. Create an instance of CosmosDBTools with the configured settings
cosmosdb_tools = CosmosDBTools(config)

# 3. Create a decorator function for registering tools
mongodb_toolbox = cosmosdb_tools.cosmosdb_toolbox

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:memorizz.database.mongodb.mongodb_tools:Vector search index 'vector_index' created.
INFO:memorizz.database.mongodb.mongodb_tools:MongoDBTools initialized successfully.


In [11]:
import random
from datetime import datetime

# 4. Define and register tool functions using the mongodb_toolbox decorator
# These functions will be stored in the MongoDB database and can be retrieved for function calling
@mongodb_toolbox()
def shout(statement: str) -> str:
  """
  Convert a statement to uppercase letters to emulate shouting. Use this when a user wants to emphasize something strongly or when they explicitly ask to 'shout' something..

  """
  return statement.upper()

@mongodb_toolbox()
def get_weather(location: str, unit: str = "celsius") -> str:
    """
    Get the current weather for a specified location.
    Use this when a user asks about the weather in a specific place.

    :param location: The name of the city or location to get weather for.
    :param unit: The temperature unit, either 'celsius' or 'fahrenheit'. Defaults to 'celsius'.
    :return: A string describing the current weather.
    """
    conditions = ["sunny", "cloudy", "rainy", "snowy"]
    temperature = random.randint(-10, 35)

    if unit.lower() == "fahrenheit":
        temperature = (temperature * 9/5) + 32

    condition = random.choice(conditions)
    return f"The weather in {location} is currently {condition} with a temperature of {temperature}°{'C' if unit.lower() == 'celsius' else 'F'}."

@mongodb_toolbox()
def get_stock_price(symbol: str) -> str:
    """
    Get the current stock price for a given stock symbol.
    Use this when a user asks about the current price of a specific stock.

    :param symbol: The stock symbol to look up (e.g., 'AAPL' for Apple Inc.).
    :return: A string with the current stock price.
    """
    price = round(random.uniform(10, 1000), 2)
    return f"The current stock price of {symbol} is ${price}."

@mongodb_toolbox()
def get_current_time(timezone: str = "UTC") -> str:
    """
    Get the current time for a specified timezone.
    Use this when a user asks about the current time in a specific timezone.

    :param timezone: The timezone to get the current time for. Defaults to 'UTC'.
    :return: A string with the current time in the specified timezone.
    """
    current_time = datetime.utcnow().strftime("%H:%M:%S")
    return f"The current time in {timezone} is {current_time}."


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:memorizz.database.mongodb.mongodb_tools:Successfully registered tool: shout
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:memorizz.database.mongodb.mongodb_tools:Successfully registered tool: get_weather
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:memorizz.database.mongodb.mongodb_tools:Successfully registered tool: get_stock_price
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:memorizz.database.mongodb.mongodb_tools:Successfully registered tool: get_current_time


In [12]:
# 5. Define the user query
# This query will be used to search for relevant tools in the MongoDB database
user_query = "Hi, can you shout the statement: We are there"

In [13]:
# 6. Populate tools based on the user query
tools = cosmosdb_tools.populate_tools(
    user_query,  # The query string to search for relevant tools
    num_tools=2  # The maximum number of tools to return from the search
    )

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:memorizz.database.mongodb.mongodb_tools:Successfully populated 2 tools


{'_id': ObjectId('677b8d78ba939fbcff8999ca'), 'name': 'shout', 'description': "Convert a statement to uppercase letters to emulate shouting. Use this when a user wants to emphasize something strongly or when they explicitly ask to 'shout' something..", 'parameters': {'type': 'object', 'properties': {'statement': {'type': 'string', 'description': 'Parameter statement'}}, 'required': ['statement'], 'additionalProperties': False}}
{'_id': ObjectId('677b8d7aba939fbcff8999cc'), 'name': 'get_stock_price', 'description': "Get the current stock price for a given stock symbol.\nUse this when a user asks about the current price of a specific stock.\n\n:param symbol: The stock symbol to look up (e.g., 'AAPL' for Apple Inc.).\n:return: A string with the current stock price.", 'parameters': {'type': 'object', 'properties': {'symbol': {'type': 'string', 'description': 'Parameter symbol'}}, 'required': ['symbol'], 'additionalProperties': False}}


In [14]:
import pprint

pprint.pprint(tools)

[{'function': {'description': 'Convert a statement to uppercase letters to '
                              'emulate shouting. Use this when a user wants to '
                              'emphasize something strongly or when they '
                              "explicitly ask to 'shout' something..",
               'name': 'shout',
               'parameters': {'additionalProperties': False,
                              'properties': {'statement': {'description': 'Parameter '
                                                                          'statement',
                                                           'type': 'string'}},
                              'required': ['statement'],
                              'type': 'object'}},
  'type': 'function'},
 {'function': {'description': 'Get the current stock price for a given stock '
                              'symbol.\n'
                              'Use this when a user asks about the current '
                      